# Inizializzazione e Caricamento Dati
In questa sezione importiamo le librerie necessarie e carichiamo il dataset del MAGIC Gamma Telescope. Il dataset contiene 10 features continue e 1 label (h=hadron, g=gamma).

In [ ]:
import os
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    recall_score, f1_score, roc_curve, roc_auc_score
)
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, adjusted_mutual_info_score
)
from sklearn.impute import SimpleImputer
from scipy import stats

warnings.filterwarnings('ignore')

# NOTE: usiamo sklearn.neural_network.MLPClassifier al posto di TensorFlow
# perché TF non supporta ancora Python 3.14.
# Le architetture sono equivalenti: NN Base = (32,16) relu adam;
# NN Pro = stessa arch + L2 (alpha) + early_stopping.
print('Librerie caricate OK')

In [ ]:
# Caricamento del dataset
col_names = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1',
             'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']

df = pd.read_csv('magic04.data', names=col_names)
df['class'] = df['class'].map({'h': 0, 'g': 1})

print('Dataset caricato con successo in locale! Dimensioni:', df.shape)

## Suddivisione del Dataset (Train, Val, Test)
Per addestrare i nostri modelli in modo corretto dividiamo il dataset in tre parti: il 60% per il Training Set, il 20% per il Validation Set e il 20% per il Test Set.

In [ ]:
X = df.drop(['class'], axis=1)
y = df['class']
seed = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=seed, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

df_train = pd.concat([X_train, y_train], axis=1)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Analisi Esplorativa
Controlliamo l'assenza di valori nulli o mancanti. Tracciamo poi istogrammi e boxplot per capire la distribuzione di ciascuna variabile e generiamo la matrice di correlazione per evidenziare relazioni lineari tra le features.

In [ ]:
print('Valori mancanti per colonna:\n', df.isnull().sum())
print(f'\nDuplicati trovati prima della pulizia: {df_train.duplicated().sum()}')

df_train = df_train.drop_duplicates()
print('Dataset Training dopo rimozione duplicati:', df_train.shape)

In [ ]:
signal_class = df_train['class'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(signal_class, labels=['gamma (1)', 'hadron (0)'],
        autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff'])
plt.title('Distribuzione del Target nel Training Set')
plt.axis('equal')
plt.show()

In [ ]:
numeric_cols = df_train.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = [c for c in numeric_cols if c != 'class']
sns.set_style('whitegrid')
n_cols = 3
n_rows = math.ceil(len(numeric_cols) / n_cols)
plt.figure(figsize=(15, 4 * n_rows))

for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.histplot(df_train[col], kde=True, bins=30, edgecolor='black')
    plt.title(f'Distribuzione di {col}')
    plt.xlabel('')
    plt.ylabel('Frequenza')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 4 * n_rows))
for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, n_cols, i + 1)
    sns.boxplot(x='class', y=col, hue='class', data=df_train,
                palette='Set2', legend=False)
    plt.title(f'{col} vs Class')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df_train[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matrice di Correlazione')
plt.show()

# Preprocessing e Scalatura
Standardizziamo i dati in modo che abbiano media 0 e deviazione standard 1. Lo scaler viene addestrato **solo sul Train** per evitare data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

y_train = y_train.astype('int64')
y_test  = y_test.astype('int64')
y_val   = y_val.astype('int64')

feature_names = list(X.columns)
print('Scaling completato.')

# Introduzione del Rumore (Noise Injection)
Implementiamo diversi tipi di sporcizia dei dati per testare la robustezza dei modelli:
- **Feature Noise (Gaussiano):** rumore gaussiano sulle features (simulazione misurazioni imprecise)
- **Label Noise Casuale:** inversione casuale uniforme delle etichette
- **Label Noise Borderline:** inversione preferenziale degli esempi vicini al confine decisionale
- **Missing Values:** introduzione di NaN in proporzione variabile
- **Outlier Injection:** inserimento di valori estremi
- **Feature Deletion (per-feature):** cancellazione selettiva di singole feature

In [ ]:
def add_gaussian_noise(X, noise_level=0.1, seed=42):
    """Aggiunge rumore gaussiano N(0, noise_level) alle features scalate."""
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=noise_level, size=X.shape)
    return X + noise


def add_label_noise_random(y, noise_level=0.1, seed=42):
    """Inversione casuale uniforme di noise_level% delle etichette."""
    rng = np.random.default_rng(seed)
    y_noisy = np.array(y).copy()
    n_flips = int(len(y_noisy) * noise_level)
    idx = rng.choice(len(y_noisy), size=n_flips, replace=False)
    y_noisy[idx] = 1 - y_noisy[idx]
    return y_noisy

# Alias retrocompatibile
add_label_noise = add_label_noise_random


def add_label_noise_borderline(X_scaled, y, noise_level=0.1, seed=42):
    """
    Inversione degli esempi più vicini al centroide opposto (borderline).
    Simula errori di etichettatura su campioni ambigui, non casuali.
    """
    rng = np.random.default_rng(seed)
    y_arr = np.array(y).copy()
    n_flips = int(len(y_arr) * noise_level)

    # distanza euclidea di ogni campione dal centroide della classe opposta
    centroids = {c: X_scaled[y_arr == c].mean(axis=0) for c in [0, 1]}
    distances = np.array([
        np.linalg.norm(X_scaled[i] - centroids[1 - y_arr[i]])
        for i in range(len(y_arr))
    ])

    # prendi i più vicini al centroide opposto (più ambigui)
    borderline_idx = np.argsort(distances)[:n_flips]
    y_arr[borderline_idx] = 1 - y_arr[borderline_idx]
    return y_arr


def add_missing_values(X, missing_rate=0.1, seed=42):
    """
    Introduce NaN casuali in missing_rate% delle celle.
    RF gestisce i missing nativamente tramite surrogate splits;
    MLP richiede imputazione.
    """
    rng = np.random.default_rng(seed)
    X_missing = X.astype(float).copy()
    mask = rng.random(X.shape) < missing_rate
    X_missing[mask] = np.nan
    return X_missing


def impute_missing(X_train_m, X_test_m, strategy='mean'):
    """Imputazione (mean/median) per MLP che non supporta NaN."""
    imp = SimpleImputer(strategy=strategy)
    return imp.fit_transform(X_train_m), imp.transform(X_test_m)


def add_outliers(X, outlier_rate=0.05, magnitude=5.0, seed=42):
    """
    Sostituisce outlier_rate% dei campioni con valori estremi
    (+/- magnitude*std) per simulare misurazioni errate.
    """
    rng = np.random.default_rng(seed)
    X_out = X.copy()
    n_out = int(len(X_out) * outlier_rate)
    idx   = rng.choice(len(X_out), size=n_out, replace=False)
    sign  = rng.choice([-1, 1], size=(n_out, X_out.shape[1]))
    X_out[idx] = sign * magnitude
    return X_out


def delete_feature(X, feature_idx):
    """Azzera completamente una feature (simula sensore guasto)."""
    X_del = X.copy()
    X_del[:, feature_idx] = 0.0
    return X_del


noise_levels = [0.1, 0.2, 0.3, 0.4]

print('Funzioni di noise definite OK')

# Modelli Neurali (MLP)
Implementiamo due varianti con `sklearn.neural_network.MLPClassifier`:
1. **Base:** architettura `(32, 16)` relu + adam, nessuna regolarizzazione.
2. **Pro (Regolarizzata):** stessa architettura + L2 (`alpha=1e-4`) + early stopping.

In [ ]:
def train_base_network(X_tr, y_tr, random_state=42):
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        solver='adam',
        max_iter=200,
        random_state=random_state,
        early_stopping=False,
        n_iter_no_change=10,
    )
    mlp.fit(X_tr, y_tr)
    return mlp


def train_pro_network(X_tr, y_tr, random_state=42):
    mlp = MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        solver='adam',
        alpha=1e-4,           # L2 regularization
        max_iter=500,
        random_state=random_state,
        early_stopping=True,  # early stopping su fraction interna
        validation_fraction=0.15,
        n_iter_no_change=15,
    )
    mlp.fit(X_tr, y_tr)
    return mlp


def evaluate_model(model, X_test_input, y_test_input, model_name='Modello'):
    print(f'\n' + '='*50)
    print(f' RISULTATI: {model_name}')
    print('='*50)
    y_pred = model.predict(X_test_input)
    y_prob = model.predict_proba(X_test_input)[:, 1]

    cm = confusion_matrix(y_test_input, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title(f'Confusion Matrix — {model_name}')
    plt.show()

    print(f'Accuracy: {accuracy_score(y_test_input, y_pred):.4f}')
    print(f'AUC:      {roc_auc_score(y_test_input, y_prob):.4f}')
    print('\nReport:')
    print(classification_report(y_test_input, y_pred, zero_division=0))
    print('='*50)


print('Definizioni modelli OK')

## Addestramento NN Base

In [ ]:
print('>>> Addestramento NN Base Baseline (Dati Puliti)')
model_nn_clean = train_base_network(X_train_scaled, y_train, random_state=seed)
evaluate_model(model_nn_clean, X_test_scaled, y_test, model_name='NN Baseline')

modelli_nn_fn   = {'Baseline': model_nn_clean}
modelli_nn_ln   = {'Baseline': model_nn_clean}
modelli_nn_comb = {'Baseline': model_nn_clean}

for lvl in noise_levels:
    print(f'\n--- NN Base RUMORE {int(lvl*100)}% ---')

    X_fn = add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=seed)
    model_fn = train_base_network(X_fn, y_train, random_state=seed)
    modelli_nn_fn[f'Rumore {int(lvl*100)}%'] = model_fn

    y_ln = add_label_noise(y_train, noise_level=lvl, seed=seed)
    model_ln = train_base_network(X_train_scaled, y_ln, random_state=seed)
    modelli_nn_ln[f'Rumore {int(lvl*100)}%'] = model_ln

    model_comb = train_base_network(X_fn, y_ln, random_state=seed)
    modelli_nn_comb[f'Rumore {int(lvl*100)}%'] = model_comb

print('\nNN Base addestrata per tutti i livelli di rumore.')

## Addestramento NN PRO (Regolarizzata)

In [ ]:
print('>>> Addestramento NN Pro Baseline (Dati Puliti)')
model_nn_clean_pro = train_pro_network(X_train_scaled, y_train, random_state=seed)
evaluate_model(model_nn_clean_pro, X_test_scaled, y_test, model_name='NN Baseline PRO')

modelli_nn_fn_pro   = {'Baseline': model_nn_clean_pro}
modelli_nn_ln_pro   = {'Baseline': model_nn_clean_pro}
modelli_nn_comb_pro = {'Baseline': model_nn_clean_pro}

for lvl in noise_levels:
    print(f'\n--- NN PRO RUMORE {int(lvl*100)}% ---')

    X_fn = add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=seed)
    model_fn_pro = train_pro_network(X_fn, y_train, random_state=seed)
    modelli_nn_fn_pro[f'Rumore {int(lvl*100)}%'] = model_fn_pro

    y_ln = add_label_noise(y_train, noise_level=lvl, seed=seed)
    model_ln_pro = train_pro_network(X_train_scaled, y_ln, random_state=seed)
    modelli_nn_ln_pro[f'Rumore {int(lvl*100)}%'] = model_ln_pro

    model_comb_pro = train_pro_network(X_fn, y_ln, random_state=seed)
    modelli_nn_comb_pro[f'Rumore {int(lvl*100)}%'] = model_comb_pro

print('\nNN Pro addestrata per tutti i livelli di rumore.')

# Intervalli di Confidenza — Analisi della Riproducibilità

Il professore solleva una domanda fondamentale: **i risultati cambiano perché i dati sono sporchi o perché cambia il seed?**

Strategia: addestriamo ogni configurazione con **N=7 seed diversi** e calcoliamo:
- Media delle metriche
- Intervallo di confidenza al 95% (IC₉₅ = x̄ ± 1.96 · σ/√N)

In questo modo distinguiamo:
1. **Varianza dovuta al seed** (rumore del training)
2. **Degradazione sistematica** dovuta alla sporcizia dei dati (segnale reale)

In [ ]:
N_RUNS   = 7
CI_SEEDS = [42, 7, 13, 99, 2024, 314, 17]

def ci_experiment(train_fn, X_tr_base, y_tr_base,
                  X_te=X_test_scaled, y_te=y_test,
                  noise_fn=None, noise_level=0.0,
                  noise_type='fn', seeds=CI_SEEDS):
    """
    Ripete l'addestramento N volte con seed diversi.
    noise_type: 'fn' (feature), 'ln' (label), 'comb' (entrambi).
    Restituisce dict con mean/std/CI di accuracy e AUC.
    """
    accs, aucs = [], []
    for s in seeds:
        X_tr = X_tr_base.copy()
        y_tr = y_tr_base.copy()

        if noise_level > 0:
            if noise_type in ('fn', 'comb'):
                X_tr = add_gaussian_noise(X_tr, noise_level=noise_level, seed=s)
            if noise_type in ('ln', 'comb'):
                y_tr = add_label_noise(y_tr, noise_level=noise_level, seed=s)

        model = train_fn(X_tr, y_tr, random_state=s)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
        accs.append(accuracy_score(y_te, y_pred))
        aucs.append(roc_auc_score(y_te, y_prob))

    def ci(vals):
        a = np.array(vals)
        m, s = a.mean(), a.std(ddof=1)
        return m, s, 1.96 * s / np.sqrt(len(a))

    acc_m, acc_s, acc_ci = ci(accs)
    auc_m, auc_s, auc_ci = ci(aucs)
    return {
        'acc_mean': acc_m, 'acc_std': acc_s, 'acc_ci': acc_ci,
        'auc_mean': auc_m, 'auc_std': auc_s, 'auc_ci': auc_ci,
    }


print('Calcolo intervalli di confidenza (NN Base, Feature Noise)...')
ci_results_base_fn = {}
for lvl in [0.0] + noise_levels:
    ci_results_base_fn[lvl] = ci_experiment(
        train_base_network, X_train_scaled, np.array(y_train),
        noise_level=lvl, noise_type='fn'
    )

print('Calcolo intervalli di confidenza (NN Base, Label Noise)...')
ci_results_base_ln = {}
for lvl in [0.0] + noise_levels:
    ci_results_base_ln[lvl] = ci_experiment(
        train_base_network, X_train_scaled, np.array(y_train),
        noise_level=lvl, noise_type='ln'
    )

print('Calcolo intervalli di confidenza (NN Pro, Feature Noise)...')
ci_results_pro_fn = {}
for lvl in [0.0] + noise_levels:
    ci_results_pro_fn[lvl] = ci_experiment(
        train_pro_network, X_train_scaled, np.array(y_train),
        noise_level=lvl, noise_type='fn'
    )

print('Calcolo intervalli di confidenza (NN Pro, Label Noise)...')
ci_results_pro_ln = {}
for lvl in [0.0] + noise_levels:
    ci_results_pro_ln[lvl] = ci_experiment(
        train_pro_network, X_train_scaled, np.array(y_train),
        noise_level=lvl, noise_type='ln'
    )

print('Intervalli di confidenza calcolati.')

In [ ]:
def plot_ci(ci_dict, metric='acc', title='', label='', color='steelblue', ax=None):
    levels  = sorted(ci_dict.keys())
    means   = [ci_dict[l][f'{metric}_mean'] for l in levels]
    ci_half = [ci_dict[l][f'{metric}_ci']   for l in levels]
    levels_pct = [l * 100 for l in levels]

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(levels_pct, means, marker='o', lw=2, color=color, label=label)
    ax.fill_between(levels_pct,
                    [m - c for m, c in zip(means, ci_half)],
                    [m + c for m, c in zip(means, ci_half)],
                    alpha=0.25, color=color)
    ax.set_xlabel('Livello di rumore (%)')
    ax.set_ylabel(metric.upper())
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    return ax


# --- Accuracy CI: Base vs Pro su Feature Noise ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_ci(ci_results_base_fn, 'acc', 'Accuracy con IC 95% — Feature Noise',
        'NN Base', '#1f77b4', axes[0])
plot_ci(ci_results_pro_fn, 'acc', 'Accuracy con IC 95% — Feature Noise',
        'NN Pro', '#d62728', axes[0])
axes[0].legend()

plot_ci(ci_results_base_ln, 'acc', 'Accuracy con IC 95% — Label Noise',
        'NN Base', '#1f77b4', axes[1])
plot_ci(ci_results_pro_ln, 'acc', 'Accuracy con IC 95% — Label Noise',
        'NN Pro', '#d62728', axes[1])
axes[1].legend()
plt.suptitle('Intervalli di Confidenza al 95% — 7 seed distinti', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Stampa tabella riassuntiva CI ---
print('\n=== TABELLA INTERVALLI DI CONFIDENZA (Accuracy) ===')
print(f'{"Modello":<15} {"Tipo":>12} {"Livello":>8} {"Media":>8} {"Std":>8} {"IC 95%":>10}')
print('-'*65)
for nome, ci_dict in [("NN Base", ci_results_base_fn), ("NN Pro", ci_results_pro_fn)]:
    for lvl in sorted(ci_dict):
        r = ci_dict[lvl]
        print(f'{nome:<15} {"Feature":>12} {int(lvl*100):>7}%'
              f' {r["acc_mean"]:>8.4f} {r["acc_std"]:>8.4f} ±{r["acc_ci"]:>8.4f}')

print('\nInterpretazione: se IC è ampio → alta varianza da seed (rumore del training).\n'
      'Se la media cala sistematicamente → degrado reale dovuto ai dati sporchi.')

# Valutazione e Confronto
Valutiamo i modelli usando Accuracy, F1 Macro, Recall Macro e AUC per capire come ciascuna tipologia di rumore ne degrada le performance.

In [ ]:
PALETTE = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e']


def compute_metrics(model, X_test_input, y_test_input):
    y_prob = model.predict_proba(X_test_input)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)
    return {
        'accuracy':     accuracy_score(y_test_input, y_pred),
        'recall_macro': recall_score(y_test_input, y_pred, average='macro', zero_division=0),
        'f1_macro':     f1_score(y_test_input, y_pred, average='macro', zero_division=0),
        'auc':          roc_auc_score(y_test_input, y_prob),
    }


def _level_from_name(name):
    if 'baseline' in name.lower(): return 0
    m = re.search(r'(\d+)', name)
    return int(m.group(1)) if m else np.nan


def results_dataframe(models_dict, X_test_input, y_test_input,
                      modello='NN base', tipo_rumore=''):
    rows = []
    for name, model in models_dict.items():
        rows.append({'modello': modello, 'tipo_rumore': tipo_rumore,
                     'livello': _level_from_name(name),
                     **compute_metrics(model, X_test_input, y_test_input)})
    return pd.DataFrame(rows).sort_values('livello').reset_index(drop=True)


df_fn   = results_dataframe(modelli_nn_fn,   X_test_scaled, y_test, 'NN base', 'Feature noise')
df_ln   = results_dataframe(modelli_nn_ln,   X_test_scaled, y_test, 'NN base', 'Label noise')
df_comb = results_dataframe(modelli_nn_comb, X_test_scaled, y_test, 'NN base', 'Combinato')

df_fn_pro   = results_dataframe(modelli_nn_fn_pro,   X_test_scaled, y_test, 'NN pro', 'Feature noise')
df_ln_pro   = results_dataframe(modelli_nn_ln_pro,   X_test_scaled, y_test, 'NN pro', 'Label noise')
df_comb_pro = results_dataframe(modelli_nn_comb_pro, X_test_scaled, y_test, 'NN pro', 'Combinato')

print('Metriche calcolate.')

In [ ]:
def plot_metrics_grid(dfs, title='Impatto del rumore sulle metriche'):
    metriche = [('accuracy','Accuracy'), ('f1_macro','F1 (macro)'),
                ('recall_macro','Recall (macro)'), ('auc','AUC')]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (key, nice) in zip(axes.ravel(), metriche):
        for (label, df_), color in zip(dfs.items(), PALETTE):
            d = df_.sort_values('livello')
            ax.plot(d['livello'], d[key], marker='o', lw=2,
                    markersize=6, color=color, label=label)
        ax.set_title(nice, fontweight='bold')
        ax.set_xlabel('Livello di rumore (%)')
        ax.set_ylabel(nice)
        ax.grid(alpha=0.3)
    axes.ravel()[0].legend(frameon=True)
    fig.suptitle(title, fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


def plot_base_vs_pro(df_base, df_pro, metric='accuracy', tipo='Feature noise'):
    m = df_base.merge(df_pro, on='livello', suffixes=('_base','_pro')).sort_values('livello')
    plt.figure(figsize=(9, 5.5))
    plt.plot(m['livello'], m[metric+'_base'], marker='o', lw=2.2,
             color=PALETTE[0], label='NN base')
    plt.plot(m['livello'], m[metric+'_pro'],  marker='s', lw=2.2,
             color=PALETTE[1], label='NN pro (regolarizzata)')
    plt.fill_between(m['livello'], m[metric+'_base'], m[metric+'_pro'],
                     alpha=0.12, color='grey')
    plt.xlabel('Livello di rumore (%)')
    plt.ylabel(metric.replace('_',' ').upper())
    plt.title(f'NN base vs NN pro — {tipo} ({metric})', fontweight='bold')
    plt.grid(alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

In [ ]:
dati_base = {'Feature noise': df_fn, 'Label noise': df_ln, 'Combinato': df_comb}
plot_metrics_grid(dati_base, title='Impatto del rumore sulle metriche (NN Base)')

plot_base_vs_pro(df_fn,   df_fn_pro,   metric='accuracy', tipo='Feature noise')
plot_base_vs_pro(df_ln,   df_ln_pro,   metric='accuracy', tipo='Label noise')
plot_base_vs_pro(df_comb, df_comb_pro, metric='accuracy', tipo='Rumore Combinato')

plot_base_vs_pro(df_fn,   df_fn_pro,   metric='f1_macro', tipo='Feature noise')
plot_base_vs_pro(df_ln,   df_ln_pro,   metric='f1_macro', tipo='Label noise')

plot_base_vs_pro(df_fn,   df_fn_pro,   metric='auc', tipo='Feature noise')
plot_base_vs_pro(df_ln,   df_ln_pro,   metric='auc', tipo='Label noise')

## Confronto Curve ROC

In [ ]:
def plot_roc_comparison(models_dict, X_test_input, y_test_input, title_suffix=''):
    plt.figure(figsize=(10, 7))
    colors = ['blue', 'green', 'orange', 'purple', 'brown']
    for i, (name, model) in enumerate(models_dict.items()):
        y_score = model.predict_proba(X_test_input)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_input, y_score)
        auc = roc_auc_score(y_test_input, y_score)
        plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
                 label=f'{name} (AUC = {auc:.4f})')

    plt.plot([0,1],[0,1], 'r--', lw=1, alpha=0.5)
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Curva ROC — {title_suffix}')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.show()


plot_roc_comparison(modelli_nn_fn,  X_test_scaled, y_test, 'Feature Noise (NN Base)')
plot_roc_comparison(modelli_nn_ln,  X_test_scaled, y_test, 'Label Noise (NN Base)')
plot_roc_comparison(modelli_nn_fn_pro, X_test_scaled, y_test, 'Feature Noise (NN Pro)')

# Modello Non Supervisionato (K-Means)

Il K-Means tenta di trovare raggruppamenti senza usare le etichette.
Il **Label Noise** non ha alcun impatto su K-Means, mentre il **Feature Noise** degrada la qualità dei cluster.

Misuriamo la qualità con:
- **ARI** (Adjusted Rand Index)
- **AMI** (Adjusted Mutual Information) — misura quanto i cluster rispecchiano le vere classi, corretta per il caso
- **Silhouette** — compattezza intra-cluster vs separazione inter-cluster
- **Purezza**

Mostriamo anche come il Feature Noise **cambia la geometria dei cluster nello spazio PCA**.

In [ ]:
def cluster_quality(X_scaled, y_true, k=2, random_state=42):
    km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    labels = km.fit_predict(X_scaled)
    y_arr = np.array(y_true)
    purity = sum(
        np.bincount(y_arr[labels == c]).max()
        for c in np.unique(labels)
    ) / len(y_arr)
    metrics = {
        'ARI':        adjusted_rand_score(y_arr, labels),
        'AMI':        adjusted_mutual_info_score(y_arr, labels),
        'NMI':        normalized_mutual_info_score(y_arr, labels),
        'purezza':    purity,
        'silhouette': silhouette_score(X_scaled, labels,
                                       sample_size=2000, random_state=random_state),
    }
    return metrics, labels, km


q_clean, labels_clean, km_clean = cluster_quality(X_train_scaled, y_train)
print('K-means (k=2) dati puliti:')
for m, v in q_clean.items():
    print(f'  {m:11s}: {v:.4f}')

print('\nInterpretazione AMI: 0 = clustering casuale, 1 = corrispondenza perfetta con le vere etichette.')
print(f'AMI={q_clean["AMI"]:.4f} → K-Means non riesce a separare bene gamma/hadron solo dalle feature.')

In [ ]:
pca2 = PCA(n_components=2, random_state=seed)
pc_clean = pca2.fit_transform(X_train_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(pc_clean[:, 0], pc_clean[:, 1], c=labels_clean,
                cmap='coolwarm', s=5, alpha=0.4)
axes[0].set_title('Cluster K-Means (k=2) — dati puliti')
axes[1].scatter(pc_clean[:, 0], pc_clean[:, 1], c=np.array(y_train),
                cmap='coolwarm', s=5, alpha=0.4)
axes[1].set_title('Classi reali (gamma=rosso / hadron=blu)')
for a in axes:
    a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.suptitle('PCA — Geometria originale (dati puliti)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analisi geometrica: come il Feature Noise cambia la disposizione PCA
noise_sigmas = [0.0, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(2, len(noise_sigmas), figsize=(16, 8))

pca_ref = PCA(n_components=2, random_state=seed)
pc_ref  = pca_ref.fit_transform(X_train_scaled)

ami_geo_vals = []

for j, sigma in enumerate(noise_sigmas):
    X_noisy = add_gaussian_noise(X_train_scaled, noise_level=sigma, seed=seed)
    pc_noisy = pca_ref.transform(X_noisy)   # stesso pca per confronto diretto

    q_n, labels_n, _ = cluster_quality(X_noisy, y_train)
    ami_geo_vals.append((sigma, q_n['AMI'], q_n['silhouette']))

    axes[0, j].scatter(pc_noisy[:, 0], pc_noisy[:, 1],
                       c=labels_n, cmap='coolwarm', s=3, alpha=0.3)
    axes[0, j].set_title(f'Cluster (σ={sigma})')
    axes[0, j].set_xlabel('PC1'); axes[0, j].set_ylabel('PC2')

    axes[1, j].scatter(pc_noisy[:, 0], pc_noisy[:, 1],
                       c=np.array(y_train), cmap='coolwarm', s=3, alpha=0.3)
    axes[1, j].set_title(f'Classi reali (σ={sigma})')
    axes[1, j].set_xlabel('PC1'); axes[1, j].set_ylabel('PC2')

plt.suptitle('Effetto del Feature Noise sulla Geometria PCA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n--- AMI e Silhouette al variare del Feature Noise ---')
print(f'{"sigma":>6}  {"AMI":>8}  {"Silhouette":>10}')
for sigma, ami_v, sil_v in ami_geo_vals:
    print(f'{sigma:>6.2f}  {ami_v:>8.4f}  {sil_v:>10.4f}')

print('\nConfronto rispetto al clustering originale (AMI e Silhouette a sigma=0):')
ami0, sil0 = ami_geo_vals[0][1], ami_geo_vals[0][2]
for sigma, ami_v, sil_v in ami_geo_vals[1:]:
    print(f'  sigma={sigma}: ΔAMI={ami_v-ami0:+.4f}  ΔSilhouette={sil_v-sil0:+.4f}')

In [ ]:
# Effetto completo Feature Noise su K-Means
feature_levels_km = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
print('--- Effetto del Feature Noise su K-Means ---')
print(f'{"sigma":>6}  {"ARI":>6}  {"AMI":>6}  {"Silhouette":>10}  {"Purezza":>8}')
for lvl in feature_levels_km:
    X_n = add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=seed)
    q, _, _ = cluster_quality(X_n, y_train)
    print(f'{lvl:>6.2f}  {q["ARI"]:>6.3f}  {q["AMI"]:>6.3f}  '
          f'{q["silhouette"]:>10.3f}  {q["purezza"]:>8.3f}')

y_train_flip = add_label_noise(y_train, noise_level=0.40, seed=seed)
_, labels_lab, _ = cluster_quality(X_train_scaled, y_train_flip)
ari_part = adjusted_rand_score(labels_clean, labels_lab)
print(f'\nARI(clustering_pulito, clustering_labelNoise40%) = {ari_part:.3f}')
print('K-Means usa solo feature → Label Noise non sposta i cluster (ARI=1.0)')

# Tipi Estesi di Sporcizia dei Dati

Il professore suggerisce di testare diversi tipi di errore oltre al semplice flip casuale:

| Tipo | Descrizione |
|------|-------------|
| Label Noise Random | Flip casuale uniforme delle etichette |
| Label Noise Borderline | Flip sugli esempi più ambigui (vicini al centroide opposto) |
| Missing Values | NaN casuali nelle features, poi imputation |
| Outlier Injection | Sostituzione di campioni con valori estremi |
| Feature Deletion | Azzeramento completo di una feature (sensore guasto) |

In [ ]:
RATE = 0.2   # 20% di sporcizia per ogni tipo

def eval_noise_type(label, X_tr, y_tr, model_fn=train_base_network):
    m = model_fn(X_tr, y_tr, random_state=seed)
    return {'label': label, **compute_metrics(m, X_test_scaled, y_test)}


rows_ext = []

# 1. Baseline
rows_ext.append(eval_noise_type('Baseline (pulito)', X_train_scaled, y_train))

# 2. Label Noise Random 20%
y_ln_r = add_label_noise_random(y_train, noise_level=RATE, seed=seed)
rows_ext.append(eval_noise_type('Label Noise Random 20%', X_train_scaled, y_ln_r))

# 3. Label Noise Borderline 20%
y_ln_b = add_label_noise_borderline(X_train_scaled, y_train, noise_level=RATE, seed=seed)
rows_ext.append(eval_noise_type('Label Noise Borderline 20%', X_train_scaled, y_ln_b))

# 4. Missing Values 20% → imputation
X_miss = add_missing_values(X_train_scaled, missing_rate=RATE, seed=seed)
X_miss_imp, X_test_imp = impute_missing(X_miss, X_test_scaled, strategy='mean')
rows_ext.append(eval_noise_type('Missing Values 20% + Imput.', X_miss_imp, y_train))

# 5. Outlier Injection 10%
X_out = add_outliers(X_train_scaled, outlier_rate=0.10, magnitude=5.0, seed=seed)
rows_ext.append(eval_noise_type('Outlier Injection 10%', X_out, y_train))

# 6. Feature Deletion: cancella fAlpha (feature 8, la più discriminante)
X_del_alpha = delete_feature(X_train_scaled, feature_idx=8)
X_test_del  = delete_feature(X_test_scaled, feature_idx=8)
m_del = train_base_network(X_del_alpha, y_train, random_state=seed)
rows_ext.append({'label': f'Feature Deletion (fAlpha)',
                 **compute_metrics(m_del, X_test_del, y_test)})

df_ext = pd.DataFrame(rows_ext)
print('\n=== CONFRONTO TIPI DI SPORCIZIA (NN Base, 20% sporcizia) ===')
print(df_ext.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Visualizzazione del confronto tra tipi di sporcizia
metrics_to_plot = ['accuracy', 'f1_macro', 'auc']
x = np.arange(len(df_ext))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, df_ext[metric], width, label=metric.upper(),
                  color=PALETTE[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(df_ext['label'], rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.6, 1.0)
ax.set_ylabel('Score')
ax.set_title('Impatto dei Diversi Tipi di Sporcizia — NN Base', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(float(df_ext.loc[df_ext['label'] == 'Baseline (pulito)', 'accuracy'].iloc[0]),
           color='black', ls='--', lw=1, alpha=0.5, label='Baseline accuracy')
plt.tight_layout()
plt.show()

print('\nDifferenza rispetto al Baseline (Accuracy):')
baseline_acc = float(df_ext.loc[df_ext['label']=='Baseline (pulito)', 'accuracy'].iloc[0])
for _, row in df_ext.iterrows():
    delta = row['accuracy'] - baseline_acc
    print(f'  {row["label"]:<35}: {delta:+.4f}')

# Random Forest — Modello Ensemble

Il Random Forest (RF) è un ensemble di alberi decisionali addestrati su sottocampioni casuali del dataset.

Punti di forza:
- **Robusto ai Missing Values** (gestisce NaN tramite surrogate splits)
- **Robustezza agli Outlier** (non è sensibile alla scala)
- **Feature Importance** interpretabile
- **Effetto di rimbalzo**: a differenza della NN, l'ensemble può mantenere performance accettabili anche con alti livelli di rumore

In [ ]:
# RF Baseline
rf_clean = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
rf_clean.fit(X_train_scaled, y_train)

print('=== Random Forest Baseline ===')
y_rf_pred = rf_clean.predict(X_test_scaled)
y_rf_prob = rf_clean.predict_proba(X_test_scaled)[:, 1]
print(f'Accuracy: {accuracy_score(y_test, y_rf_pred):.4f}')
print(f'AUC:      {roc_auc_score(y_test, y_rf_prob):.4f}')
print(classification_report(y_test, y_rf_pred, zero_division=0))

# Feature importance
fi = pd.Series(rf_clean.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=fi.values, y=fi.index, palette='viridis')
plt.title('RF — Feature Importance (dati puliti)', fontweight='bold')
plt.xlabel('Importanza media')
plt.tight_layout()
plt.show()

In [ ]:
# RF con tutti i livelli di rumore
modelli_rf_fn   = {'Baseline': rf_clean}
modelli_rf_ln   = {'Baseline': rf_clean}
modelli_rf_comb = {'Baseline': rf_clean}

for lvl in noise_levels:
    X_fn = add_gaussian_noise(X_train_scaled, noise_level=lvl, seed=seed)
    rf_fn = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    rf_fn.fit(X_fn, y_train)
    modelli_rf_fn[f'Rumore {int(lvl*100)}%'] = rf_fn

    y_ln = add_label_noise(y_train, noise_level=lvl, seed=seed)
    rf_ln = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    rf_ln.fit(X_train_scaled, y_ln)
    modelli_rf_ln[f'Rumore {int(lvl*100)}%'] = rf_ln

    rf_comb = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    rf_comb.fit(X_fn, y_ln)
    modelli_rf_comb[f'Rumore {int(lvl*100)}%'] = rf_comb

print('RF addestrato per tutti i livelli di rumore.')


def compute_metrics_rf(model, X_te, y_te):
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)
    return {
        'accuracy':     accuracy_score(y_te, y_pred),
        'recall_macro': recall_score(y_te, y_pred, average='macro', zero_division=0),
        'f1_macro':     f1_score(y_te, y_pred, average='macro', zero_division=0),
        'auc':          roc_auc_score(y_te, y_prob),
    }

df_rf_fn   = results_dataframe(modelli_rf_fn,   X_test_scaled, y_test, 'RF', 'Feature noise')
df_rf_ln   = results_dataframe(modelli_rf_ln,   X_test_scaled, y_test, 'RF', 'Label noise')
df_rf_comb = results_dataframe(modelli_rf_comb, X_test_scaled, y_test, 'RF', 'Combinato')

print('Metriche RF calcolate.')

In [ ]:
# Confronto RF vs NN Base vs NN Pro
def plot_three_models(df_nn_base, df_nn_pro, df_rf, metric='accuracy',
                      tipo='Feature noise', title=None):
    plt.figure(figsize=(10, 5.5))
    for df_, color, label, mark in [
        (df_nn_base, PALETTE[0], 'NN Base', 'o'),
        (df_nn_pro,  PALETTE[1], 'NN Pro',  's'),
        (df_rf,      PALETTE[2], 'RF',      '^'),
    ]:
        d = df_.sort_values('livello')
        plt.plot(d['livello'], d[metric], marker=mark, lw=2.2,
                 color=color, label=label, markersize=8)
    plt.xlabel('Livello di rumore (%)')
    plt.ylabel(metric.replace('_',' ').upper())
    plt.title(title or f'RF vs NN — {tipo} ({metric})', fontweight='bold')
    plt.grid(alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()


plot_three_models(df_fn, df_fn_pro, df_rf_fn, 'accuracy', 'Feature noise')
plot_three_models(df_ln, df_ln_pro, df_rf_ln, 'accuracy', 'Label noise')
plot_three_models(df_comb, df_comb_pro, df_rf_comb, 'accuracy', 'Rumore Combinato')

plot_three_models(df_fn, df_fn_pro, df_rf_fn, 'auc', 'Feature noise')
plot_three_models(df_ln, df_ln_pro, df_rf_ln, 'auc', 'Label noise')

plot_three_models(df_fn, df_fn_pro, df_rf_fn, 'f1_macro', 'Feature noise')

In [ ]:
# RF vs MLP con Missing Values
# RF gestisce NaN nativamente; MLP richiede imputation
missing_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]

acc_rf_miss   = []
acc_mlp_miss  = []

for rate in missing_rates:
    X_miss = add_missing_values(X_train_scaled, missing_rate=rate, seed=seed)
    X_miss_te = add_missing_values(X_test_scaled, missing_rate=rate, seed=seed+1)

    # RF: passa i NaN direttamente (sklearn RF gestisce missing natively)
    rf_m = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    try:
        rf_m.fit(X_miss, y_train)
        acc_rf = accuracy_score(y_test, rf_m.predict(X_miss_te))
    except Exception:
        # sklearn RF NON supporta NaN nativamente; usiamo imputazione anche per RF
        X_m_imp, X_te_imp = impute_missing(X_miss, X_miss_te, strategy='median')
        rf_m.fit(X_m_imp, y_train)
        acc_rf = accuracy_score(y_test, rf_m.predict(X_te_imp))
    acc_rf_miss.append(acc_rf)

    # MLP: richiede imputation mean
    X_m_imp_mlp, X_te_imp_mlp = impute_missing(X_miss, X_miss_te, strategy='mean')
    mlp_m = train_base_network(X_m_imp_mlp, y_train, random_state=seed)
    acc_mlp = accuracy_score(y_test, mlp_m.predict(X_te_imp_mlp))
    acc_mlp_miss.append(acc_mlp)

plt.figure(figsize=(9, 5.5))
plt.plot([r*100 for r in missing_rates], acc_rf_miss,
         marker='^', lw=2.2, color=PALETTE[2], label='RF (imputazione mediana)')
plt.plot([r*100 for r in missing_rates], acc_mlp_miss,
         marker='o', lw=2.2, color=PALETTE[0], label='NN Base (imputazione media)')
plt.xlabel('Missing rate (%)')
plt.ylabel('Accuracy')
plt.title('Effetto dei Missing Values — RF vs NN Base', fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print('\n--- Accuracy su Missing Values ---')
print(f'{"Rate":>6}  {"RF":>8}  {"MLP":>8}')
for r, a_rf, a_mlp in zip(missing_rates, acc_rf_miss, acc_mlp_miss):
    print(f'{r*100:>5}%  {a_rf:>8.4f}  {a_mlp:>8.4f}')

In [ ]:
# Effetto di 'rimbalzo' (bounce-back): cosa succede ai modelli quando
# addestriamo su dati sporchi ma testiamo su dati puliti vs sporchi?
# Verifichiamo anche l'effetto di 'ripulitura' (correzione dell'etichetta borderline).

BOUNCE_LEVEL = 0.3

# Scenario A: addestro con label noise 30%, testo su test set pulito
y_ln_30 = add_label_noise_random(y_train, noise_level=BOUNCE_LEVEL, seed=seed)
rf_dirty  = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
rf_dirty.fit(X_train_scaled, y_ln_30)

nn_dirty  = train_base_network(X_train_scaled, y_ln_30, random_state=seed)
nn_pro_d  = train_pro_network(X_train_scaled, y_ln_30, random_state=seed)

# Scenario B: training su dati sporchi borderline 30%
y_border_30 = add_label_noise_borderline(X_train_scaled, y_train,
                                          noise_level=BOUNCE_LEVEL, seed=seed)
rf_border = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
rf_border.fit(X_train_scaled, y_border_30)
nn_border = train_base_network(X_train_scaled, y_border_30, random_state=seed)

scenarios = [
    ('Baseline', rf_clean,    model_nn_clean,  model_nn_clean_pro),
    (f'Label Noise Random {int(BOUNCE_LEVEL*100)}%',  rf_dirty,   nn_dirty,   nn_pro_d),
    (f'Label Noise Borderline {int(BOUNCE_LEVEL*100)}%', rf_border, nn_border, None),
]

print('\n=== EFFETTO DI RIMBALZO (BOUNCE-BACK) ===')
print(f'{"Scenario":<36}  {"RF acc":>8}  {"NN Base acc":>11}  {"NN Pro acc":>10}')
print('-'*72)
for nome, rf_m, nn_m, nn_pro_m in scenarios:
    acc_rf = accuracy_score(y_test, rf_m.predict(X_test_scaled))
    acc_nn = accuracy_score(y_test, nn_m.predict(X_test_scaled))
    acc_pro = accuracy_score(y_test, nn_pro_m.predict(X_test_scaled)) if nn_pro_m else float('nan')
    print(f'{nome:<36}  {acc_rf:>8.4f}  {acc_nn:>11.4f}  {acc_pro:>10.4f}')

print('\nInterpretazione: il label noise borderline è più "insidioso" del random\n'
      'perché corrompe gli esempi più vicini al confine decisionale.')

# Tabella Riassuntiva Finale

Raccogliamo le performance al **40% di rumore** (caso peggiore) per ogni modello e tipo di sporcizia.

In [ ]:
def get_worst_case(df_noise, metric='accuracy', worst_level=40):
    row = df_noise[df_noise['livello'] == worst_level]
    return float(row[metric].iloc[0]) if len(row) else float('nan')

def get_baseline(df_noise, metric='accuracy'):
    row = df_noise[df_noise['livello'] == 0]
    return float(row[metric].iloc[0]) if len(row) else float('nan')


summary_rows = []
for tipo, df_nn, df_pro, df_rf in [
    ('Feature Noise', df_fn,   df_fn_pro,   df_rf_fn),
    ('Label Noise',   df_ln,   df_ln_pro,   df_rf_ln),
    ('Combinato',     df_comb, df_comb_pro, df_rf_comb),
]:
    for metrica in ['accuracy', 'auc', 'f1_macro']:
        bl_nn  = get_baseline(df_nn,  metrica)
        bl_pro = get_baseline(df_pro, metrica)
        bl_rf  = get_baseline(df_rf,  metrica)
        wc_nn  = get_worst_case(df_nn,  metrica)
        wc_pro = get_worst_case(df_pro, metrica)
        wc_rf  = get_worst_case(df_rf,  metrica)
        summary_rows.append({
            'Tipo Rumore': tipo, 'Metrica': metrica,
            'NN Base @0%': f'{bl_nn:.4f}', 'NN Base @40%': f'{wc_nn:.4f}',
            'NN Pro @0%':  f'{bl_pro:.4f}', 'NN Pro @40%':  f'{wc_pro:.4f}',
            'RF @0%':      f'{bl_rf:.4f}',  'RF @40%':       f'{wc_rf:.4f}',
        })

df_summary = pd.DataFrame(summary_rows)
print('=== TABELLA RIASSUNTIVA — Baseline vs Caso Peggiore (40% rumore) ===')
print(df_summary.to_string(index=False))

# Salva anche come CSV
df_summary.to_csv('risultati_summary.csv', index=False)
print('\nSalvata anche in risultati_summary.csv')

In [ ]:
# ROC finale: tutti e tre i modelli a dati puliti e a 40% feature noise
plt.figure(figsize=(10, 7))
pairs = [
    ('NN Base (pulito)', model_nn_clean,     'blue',   '-'),
    ('NN Pro (pulito)',  model_nn_clean_pro,  'green',  '-'),
    ('RF (pulito)',      rf_clean,            'red',    '-'),
    ('NN Base 40%FN',   modelli_nn_fn['Rumore 40%'],    'blue',   '--'),
    ('NN Pro 40%FN',    modelli_nn_fn_pro['Rumore 40%'], 'green', '--'),
    ('RF 40%FN',        modelli_rf_fn['Rumore 40%'],     'red',   '--'),
]
for name, model, color, ls in pairs:
    y_score = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc = roc_auc_score(y_test, y_score)
    plt.plot(fpr, tpr, color=color, lw=2, ls=ls, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC — Tutti i Modelli: Dati Puliti vs 40% Feature Noise', fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()